In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import skew
import statsmodels.api as sm
from statsmodels.formula.api import ols

from config import SMALL_CATEGORY_THRESHOLD
from utils import collapse_small_categories, load_star_schema

df = load_star_schema()

df["log_rating_count"] = np.log1p(df["rating_count"])
df["log_n_reviewers"] = np.log1p(df["n_reviewers"])

print(df.shape)


(1350, 13)


In [2]:
# Confirmation about skew

print("rating_count skewness:", skew(df["rating_count"]))
print("n_reviewers skewness:", skew(df["n_reviewers"]))


rating_count skewness: 5.684577144067881
n_reviewers skewness: 3.4606930813833094


In [3]:
# Category group sizes
counts = df["category_main"].value_counts()
print(counts)

small_categories = counts[counts < SMALL_CATEGORY_THRESHOLD].index.tolist()
print(
    f"\nCategories with <{SMALL_CATEGORY_THRESHOLD} products (unstable if used as-is):",
    small_categories,
)


category_main
Electronics              490
Home&Kitchen             447
Computers&Accessories    375
OfficeProducts            31
MusicalInstruments         2
HomeImprovement            2
Toys&Games                 1
Health&PersonalCare        1
Car&Motorbike              1
Name: count, dtype: int64

Categories with <10 products (unstable if used as-is): ['MusicalInstruments', 'HomeImprovement', 'Toys&Games', 'Health&PersonalCare', 'Car&Motorbike']


In [4]:
df = collapse_small_categories(df)
if small_categories:
    print("\nModified distribution:\n", df["category_main"].value_counts())



Modified distribution:
 category_main
Electronics              490
Home&Kitchen             447
Computers&Accessories    375
OfficeProducts            31
Other                      7
Name: count, dtype: int64


In [5]:
# Naive relationship (Pearson + Spearman)

for target in ['rating', 'log_rating_count', 'log_n_reviewers']:
    r_p, p_p = stats.pearsonr(df['discount_percentage'], df[target])
    r_s, p_s = stats.spearmanr(df['discount_percentage'], df[target])
    print(f"{target:18s} | Pearson r={r_p:+.3f} (p={p_p:.4f})  Spearman r={r_s:+.3f} (p={p_s:.4f})")

rating             | Pearson r=-0.162 (p=0.0000)  Spearman r=-0.150 (p=0.0000)
log_rating_count   | Pearson r=-0.141 (p=0.0000)  Spearman r=-0.135 (p=0.0000)
log_n_reviewers    | Pearson r=+0.123 (p=0.0000)  Spearman r=+0.145 (p=0.0000)


In [6]:
# Anova (Prove the confound exists)

anova_model = ols('discount_percentage ~ C(category_main)', data=df).fit()
anova_table = sm.stats.anova_lm(anova_model, typ=2)
print(anova_table)

                     sum_sq      df          F        PR(>F)
C(category_main)   7.668780     4.0  46.534766  1.151298e-36
Residual          55.412921  1345.0        NaN           NaN


In [7]:
# Robust standard errors (naive vs controlled)

outcomes = ["rating", "log_rating_count", "log_n_reviewers"]
results = []
controlled_models = {}

for outcome in outcomes:
    naive = ols(f"{outcome} ~ discount_percentage", data=df).fit(cov_type="HC3")
    controlled = ols(
        f"{outcome} ~ discount_percentage + C(category_main) + C(price_tier)",
        data=df,
    ).fit(cov_type="HC3")
    controlled_models[outcome] = controlled

    naive_coef = naive.params["discount_percentage"]
    naive_p = naive.pvalues["discount_percentage"]
    ctrl_coef = controlled.params["discount_percentage"]
    ctrl_p = controlled.pvalues["discount_percentage"]
    # positive = strengthened after controls; negative = weakened (partly confound-explained)
    pct_change_in_magnitude = (abs(ctrl_coef) - abs(naive_coef)) / abs(naive_coef) * 100

    results.append({
        "outcome": outcome,
        "naive_coef": round(naive_coef, 4),
        "naive_p": round(naive_p, 4),
        "controlled_coef": round(ctrl_coef, 4),
        "controlled_p": round(ctrl_p, 4),
        "pct_change_in_coef": round(pct_change_in_magnitude, 1),
        "sign_flipped": np.sign(naive_coef) != np.sign(ctrl_coef),
    })

print(pd.DataFrame(results))

# Full diagnostic detail for the flagship outcome only
print(controlled_models["rating"].summary())


            outcome  naive_coef  naive_p  controlled_coef  controlled_p  \
0            rating     -0.2226      0.0          -0.2603        0.0000   
1  log_rating_count     -1.3390      0.0          -2.1526        0.0000   
2   log_n_reviewers      0.1393      0.0           0.1054        0.0008   

   pct_change_in_coef  sign_flipped  
0                16.9         False  
1                60.8         False  
2               -24.3         False  
                            OLS Regression Results                            
Dep. Variable:                 rating   R-squared:                       0.079
Model:                            OLS   Adj. R-squared:                  0.074
Method:                 Least Squares   F-statistic:                     15.42
Date:                Sun, 20 Sep 2026   Prob (F-statistic):           7.54e-22
Time:                        01:57:10   Log-Likelihood:                -222.37
No. Observations:                1350   AIC:                             

## Findings

- **Naive correlation:** discount % vs. rating — r = −0.162 (p<0.0001); vs. log(rating_count) — r = −0.141 (p<0.0001); vs. log(n_reviewers) — r = +0.123 (p<0.0001)
- **Confound check:** discount % varies significantly by category (F = 46.5, p ≈ 1.15e-36) — deeper discounts are not randomly distributed across product categories
- **Controlled model:** after adjusting for category and price tier, the discount–rating relationship *strengthened* by 17% (coef −0.223 → −0.260, p<0.0001) and the discount–log(rating_count) relationship strengthened by 61% (coef −1.339 → −2.153, p<0.0001). The discount–log(n_reviewers) relationship weakened by 24% but remained significant (coef +0.139 → +0.105, p=0.0008).
- **Conclusion:** Deeper discounting is associated with lower ratings and lower cumulative rating volume, independent of category and price tier — and this relationship is not explained away by confounding category composition; if anything, category composition was partly masking it. The one divergence — a positive relationship with this snapshot's reviewer count — suggests discounted products draw more *reviewers in the current scrape window* even as their *lifetime* rating volume and satisfaction run lower; this discrepancy is reported rather than resolved, since a static cross-sectional dataset can't distinguish cause from correlation here.